# Task 2 — Yelp Polarity sentiment classification (no pretrained embeddings)
**DATA266 Lab 1 · Member: Shibin Thomas**

The logic lives in tested modules next to this notebook (`data_prep.py`, `models.py`, `train.py`,
`evaluate.py`, `error_analysis.py`); the notebook calls them, so it gives the same results as the
one-command CLI `python task2_sentiment/shibin_thomas/src/run_pipeline.py`.

| Brief section | Where |
|---|---|
| 2.1 Length / class distribution, balance; missing & malformed entries | §1 |
| 2.1 Lowercasing, punctuation, stopwords, stemming; tokenisation; embeddings learned from scratch | §2 |
| 2.2 Baseline + 2 experimental models, architecture & embedding justification | §3, `results.md` |
| 2.2 Full metrics per model; exact CPU/GPU | §4–5 |
| 2.2 Manual review of 20 errors | §6, `failure_analysis.md` |
| 2.3 Comparison, strengths / weaknesses / limitations, future work | §5, `results.md` |

**No pretrained embeddings or language models are used** — every model starts from a randomly initialised `nn.Embedding`.

In [ ]:
# ---- Settings -------------------------------------------------------------------------------
MODEL_CONFIGS = ["task2_sentiment/shibin_thomas/configs/baseline_meanpool.yaml",
                 "task2_sentiment/shibin_thomas/configs/exp1_textcnn.yaml",
                 "task2_sentiment/shibin_thomas/configs/exp2_bigru_attn.yaml"]
SMOKE = False            # True -> tiny data, 1 epoch (pipeline check only)
OVERRIDES = []           # e.g. ["training.batch_size=128"]
RUN_TRAINING = "auto"    # "auto": train only if no finished run exists; True / False to force
RUN_ID = "latest"

In [ ]:
import sys, json
from pathlib import Path
here = Path.cwd().resolve()
REPO = next((p for p in [here, *here.parents] if (p / "task2_sentiment").is_dir()), None)
assert REPO is not None, "open this notebook from inside the cloned repository"
SRC = REPO / "task2_sentiment" / "shibin_thomas" / "src"
sys.path.insert(0, str(SRC))

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from utils import MEMBER_DIR, read_json, repo_path, run_dirs, get_device, hardware_info
from train import load_model_config
from data_prep import prepare_data, Preprocessor, load_vocab
from models import build_model, count_params

overrides = ([f"data_config=task2_sentiment/shibin_thomas/configs/data_smoke.yaml", "training.epochs=1",
              "smoke=true"] if SMOKE else []) + OVERRIDES
cfgs = [load_model_config(p, overrides) for p in MODEL_CONFIGS]
device = get_device("auto")
print("device:", device, "|", hardware_info(device).get("gpu_name") or hardware_info(device)["cpu_model"])
print("torch", torch.__version__)

## 1. Data analysis, missing values and malformed entries (2.1.1–2.1.2)
The official **test** split (38K reviews) is the test set for every team member; my **validation** split
(50K, stratified, seed 266) is carved out of the official train split (560K).
Malformed rows (missing / non-string text, invalid label, empty after normalising the Yelp dump artefacts)
are dropped; exact duplicates inside train and train reviews that also occur in test (leakage) are removed.

In [ ]:
processed = prepare_data(cfgs[0]["data"])
meta = read_json(processed / "meta.json")
print("Malformed / missing report:")
display(pd.DataFrame(meta["malformed_report"]).fillna(0).astype(int))
eda = meta["eda"]
rows = []
for split in ("train_full", "train", "val", "test"):
    e = eda[split]
    rows.append({"split": split, "reviews": e["n"], "negative": e["class_counts"]["negative"],
                 "positive": e["class_counts"]["positive"], "positive rate": round(e["positive_rate"], 4),
                 "balance (min/max)": round(e["balance_ratio_minority_over_majority"], 4),
                 "words mean": round(e["length_words"]["mean"], 1), "words median": e["length_words"]["median"],
                 "words p95": e["length_words"]["p95"], "words max": e["length_words"]["max"]})
display(pd.DataFrame(rows).set_index("split"))
print("mean length (words) by class, train:", eda["train"]["length_words_negative"], eda["train"]["length_words_positive"])

In [ ]:
eda_dir = MEMBER_DIR / "outputs" / "eda" / Path(cfgs[0]["data"]["processed_dir"]).name
for f in meta["eda_plots"]:
    display(Image(filename=str(eda_dir / f)))

## 2. Text preprocessing, tokenisation and embeddings (2.1.3–2.1.5)
Steps: normalise dump artefacts → **lowercase** → expand contractions (*didn't → did not*) →
**remove punctuation & special characters** → **remove stopwords** (scikit-learn list, but negations and
contrast words such as *not, no, never, but, however* are deliberately **kept** — dropping *not* would turn
"not good" into "good") → **Snowball stemming** → whitespace **tokenisation** → vocabulary from my training
split (min frequency 3, 30K types) → ids, padded / truncated to 256 tokens with head+tail truncation.
The ids feed an `nn.Embedding` (128-d) that every model **learns from scratch**.

In [ ]:
pre = Preprocessor(cfgs[0]["data"])
demo = "The pizza wasn't GOOD at all... but the staff were AMAZING!!! I'd give it 3 stars &amp; come back :)"
display(pd.DataFrame(pre.steps(demo).items(), columns=["step", "text"]))
display(pd.DataFrame(meta["preprocessing_examples"]).T)

In [ ]:
itos = load_vocab(processed)
s = meta["splits"]
print(f"vocabulary: {len(itos):,} tokens (0=<pad>, 1=<unk>); stopwords removed: {meta['stopwords_removed']}")
display(pd.DataFrame(s).T[["n", "tokens_mean", "tokens_median", "tokens_p95", "truncated_fraction",
                           "oov_token_rate", "empty_after_preprocessing"]])
print("most negative tokens:", eda["most_negative_tokens"][:15])
print("most positive tokens:", eda["most_positive_tokens"][:15])
X = np.load(processed / "X_train.npy", mmap_mode="r"); L = np.load(processed / "len_train.npy")
print("\nexample encoded review:", X[0, :min(20, L[0])].tolist(), "->", " ".join(itos[i] for i in X[0, :min(20, L[0])]))

## 3. Models (2.2.1–2.2.2)
| | Model | Idea | Why |
|---|---|---|---|
| Baseline | **MeanPoolMLP** | masked mean of learned embeddings → MLP | bag-of-words: fast, strong, order-insensitive reference |
| Experiment 1 | **TextCNN** (3/4/5-gram filters, max-over-time) | detects local phrases | captures "not good", "highly recommend" that a bag of words cannot |
| Experiment 2 | **BiGRU + attention** (2 layers) | reads the whole review in order; attention pools | long-range context, contrast ("…but the food was great") |

All three share the same data, vocabulary and a **128-d embedding learned from scratch**, so differences come
from the architecture. Full justification: `results.md`.

In [ ]:
vocab_size = len(itos)
for c in cfgs:
    m = build_model(c["model"], vocab_size)
    print(f"--- {c['name']} ({c.get('role')}) : {count_params(m):,} parameters")
    print(m)

## 4. Training (early stopping on validation macro-F1)
BCE loss on one logit, AdamW, linear warm-up + cosine decay, gradient clipping, early stopping (patience 2);
the best validation epoch is kept. Logs: `reproducibility/raw_logs/task2_sentiment/shibin_thomas/<run_id>/<model>/`.

In [ ]:
from run_pipeline import run_pipeline
prefix = "smoke_yelp3_" if SMOKE else "yelp3_"
names = [c["name"] for c in cfgs]
finished = sorted(p.name for p in (MEMBER_DIR / "outputs").glob(prefix + "*")
                  if all((p / n / "train_summary.json").exists() for n in names) and (p / "comparison.md").exists())
if RUN_TRAINING == "auto":
    RUN_TRAINING = not finished
if RUN_TRAINING:
    RUN_ID = run_pipeline(MODEL_CONFIGS, OVERRIDES, smoke=SMOKE)
elif RUN_ID == "latest":
    RUN_ID = finished[-1]
dirs = run_dirs(RUN_ID)
print("trained now" if RUN_TRAINING else "re-using finished run", "->", RUN_ID)
for n in names:
    print(f"\n{n}:")
    display(pd.read_csv(dirs["raw_logs"] / n / "epochs.csv"))

## 5. Evaluation on the official test split (2.2.3) and comparison (2.3)
All metrics come from the team's shared definitions in `task2_sentiment/shared_eval/metrics.py`.

In [ ]:
display(Markdown((dirs["outputs"] / "comparison.md").read_text(encoding="utf-8")))

In [ ]:
for f in ["training_curves.png", "roc_curves.png", "pr_curves.png", "reliability.png", "slice_error_rates.png"]:
    display(Image(filename=str(dirs["outputs"] / f)))
for n in names:
    display(Image(filename=str(dirs["outputs"] / n / "confusion_matrix.png")))

In [ ]:
mc = read_json(dirs["outputs"] / "mcnemar.json")
print("Paired McNemar test, baseline =", mc["baseline"])
display(pd.DataFrame(mc["tests"]).T)
hw = {n: read_json(dirs["outputs"] / n / "train_summary.json")["hardware"] for n in names}
print("Exact hardware per model:")
display(pd.DataFrame(hw).T)

In [ ]:
# attention weights of the BiGRU on one test review: which words does it look at?
from evaluate import load_trained
gru_name = next((c["name"] for c in cfgs if c["model"]["type"] == "bigru_attn"), None)
if gru_name:
    model, _ = load_trained(dirs["checkpoints"] / gru_name / "best_model.pt", torch.device("cpu"))
    Xt = np.load(processed / "X_test.npy", mmap_mode="r"); Lt = np.load(processed / "len_test.npy")
    i = int(np.argmin(np.abs(Lt - 40)))
    ids = torch.tensor(np.array(Xt[i:i + 1, :Lt[i]]), dtype=torch.long)
    with torch.no_grad():
        prob = torch.sigmoid(model(ids, torch.tensor([Lt[i]]))).item()
    a = model.last_attention[0].numpy()
    toks = [itos[t] for t in ids[0].tolist()]
    fig, ax = plt.subplots(figsize=(12, 2.2))
    ax.bar(range(len(toks)), a, color="#1f5aa6"); ax.set_xticks(range(len(toks)), toks, rotation=90, fontsize=8)
    ax.set(title=f"BiGRU attention on test review {i} (P(pos) = {prob:.3f})", ylabel="weight")
    fig.tight_layout(); display(fig); plt.close(fig)

## 6. Manual review of 20 errors (2.2.4)
`error_analysis.py` takes my best model (by validation macro-F1) and selects 5 confident false positives,
5 confident false negatives, 5 near-threshold errors and 5 failures from the slice with the highest error rate.
I read each review, confirmed or corrected the error type and proposed a testable fix in
[`failure_analysis.md`](../failure_analysis.md).

In [ ]:
fa = MEMBER_DIR / "failure_analysis.md"
src = fa if (fa.exists() and not SMOKE) else dirs["outputs"] / "error_candidates.md"
display(Markdown(src.read_text(encoding="utf-8")))

## 7. Evidence trail
* Raw logs (unedited): `reproducibility/raw_logs/task2_sentiment/shibin_thomas/<run_id>/`
* Manifest (versions, hardware, git commit, config hashes, checkpoint SHA-256): `reproducibility/manifests/task2_sentiment/shibin_thomas/<run_id>.json`
* Checkpoints: `task2_sentiment/shibin_thomas/checkpoints/<run_id>/<model>/best_model.pt`
* All metrics: `metrics_report.csv` · comparison: `comparison.md` · write-up: `results.md`, `failure_analysis.md`